# 실습 4 · 선형 회귀 직접 만들기

**인공지능 일반**

광고비로 매출을 예측하는 직선을 **코드로** 찾습니다.
지도 학습의 네 단계를 그대로 따라갑니다.

| 순서 | 내용 |
|---|---|
| 1 | 데이터 만들기 — 훈련 / 테스트 나누기 |
| 2 | STEP 1 · 모델의 형태 — 직선 |
| 3 | STEP 2 · 채점 기준 — RSS |
| 4 | STEP 3-① · 공식으로 한 번에 (+ 행렬로: 정규방정식) |
| 5 | STEP 3-② · 경사하강법으로 조금씩 |
| 6 | STEP 4 · 예측과 평가 — RMSE, R² |
| 7 | 도전 과제 |

> 셀 실행은 `Shift + Enter`. 위에서부터 **순서대로** 실행하세요.

## ✏️ 빈칸 6곳

코드에 `여기를_채우세요` 가 ①~⑥ 여섯 군데 있습니다(④는 a, b 두 줄). 주석의 힌트를 보고 채운 뒤 실행하세요.

| 번호 | 위치 | 무엇을 |
|---|---|---|
| ① | STEP 1 | 직선 모델 `ŷ = a·x + b` |
| ② | STEP 2 | RSS — 오차 제곱의 합 |
| ③ | STEP 3-① | 기울기 `a` 공식 |
| ④ | STEP 3-② | 경사하강법 한 걸음 |
| ⑤ | STEP 4 | R² = 1 − RSS / TSS |
| ⑥ | STEP 3-① 행렬 | 정규방정식 `β = (XᵀX)⁻¹XᵀY` |

> 빈칸을 채우지 않고 실행하면 `NameError: name '여기를_채우세요' is not defined` 가 납니다.

> 맨 아래 도전 과제에도 빈칸이 있습니다 (✏️ 도전 1~5).

## 0. 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('준비 완료!')

## 1. 데이터 만들기

어떤 회사가 지역 100곳에서 **TV 광고비**와 **매출**을 기록했다고 하겠습니다.

- `x` : TV 광고비 (단위: 백만 원, 0 ~ 3)
- `y` : 매출 (단위: 억 원)

실제 관계는 `y = 7 + 4.8·x + ε` 로 정해 두었습니다. ε은 날씨, 경쟁사 같은 **잡음**입니다.
우리 모델은 이 식을 **모르는 채로** 데이터만 보고 기울기와 절편을 찾아야 합니다.

데이터의 80%는 **훈련용**, 20%는 **테스트용**으로 나눕니다. 테스트 데이터는 마지막 평가 때까지 쓰지 않습니다.

In [ ]:
rng = np.random.default_rng(0)          # 실행할 때마다 같은 데이터가 나오도록 고정

N = 100
x = rng.uniform(0, 3, N)                # TV 광고비
noise = rng.normal(0, 1.2, N)           # 잡음 ε
y = 7 + 4.8 * x + noise                 # 매출

# 훈련 80개 / 테스트 20개로 나누기
idx = rng.permutation(N)
train, test = idx[:80], idx[80:]
x_train, y_train = x[train], y[train]
x_test,  y_test  = x[test],  y[test]

print('훈련 데이터:', len(x_train), '개   테스트 데이터:', len(x_test), '개')
print('처음 5개 (광고비, 매출):')
for k in range(5):
    print(f'  {x_train[k]:.2f}  →  {y_train[k]:.2f}')

plt.scatter(x_train, y_train, s=18, label='훈련')
plt.scatter(x_test,  y_test,  s=18, marker='s', label='테스트')
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)')
plt.legend(); plt.title('광고비와 매출'); plt.show()

## 2. STEP 1 · 모델의 형태 — 직선

모델은 직선 `ŷ = a·x + b` 로 정합니다.

- `a` (기울기), `b` (절편)은 아직 **모릅니다**. 데이터로 정할 값(파라미터)입니다.
- 사람이 정하는 것은 "직선이다"라는 **모양**뿐입니다.

In [ ]:
def predict(x, a, b):
    # ✏️ 실습 ① : 직선 모델 ŷ = a·x + b 를 계산해 돌려준다
    return 여기를_채우세요


# 확인: a=2, b=1 이면 x=3 → 7
print(predict(3, 2, 1))
print(predict(np.array([0, 1, 2]), 2, 1))   # 배열을 넣으면 한꺼번에 계산

## 3. STEP 2 · 채점 기준 — RSS

**RSS (잔차 제곱합)** = (정답 − 예측)²을 모두 더한 값

$$\text{RSS} = \sum_{i=1}^{n}(y_i - \hat y_i)^2$$

작을수록 좋은 모델입니다. 먼저 손으로 계산했던 데이터 4개로 맞는지 확인합니다.

| x | y | ŷ = 2x | 오차 | 오차² |
|---|---|---|---|---|
| 1 | 3 | 2 | +1 | 1 |
| 2 | 5 | 4 | +1 | 1 |
| 3 | 4 | 6 | −2 | 4 |
| 4 | 8 | 8 | 0 | 0 |

손 계산 결과: RSS = 1 + 1 + 4 + 0 = **6**

RSS를 데이터 개수 n으로 나눈 평균을 **MSE**(평균 제곱 오차)라고 합니다. 여기서는 6 / 4 = **1.5**입니다.
둘 중 무엇을 최소로 만들어도 찾는 `a`, `b`는 같습니다. 개수가 다른 데이터끼리(훈련 80개, 테스트 20개) 비교할 때는 MSE를 씁니다.

In [ ]:
def rss(y, y_hat):
    # ✏️ 실습 ② : (정답 − 예측)² 의 합   힌트: np.sum( (…) ** 2 )
    return 여기를_채우세요


def mse(y, y_hat):
    return rss(y, y_hat) / len(y)     # RSS를 데이터 개수로 나눈 평균


# 손 계산과 같은지 확인 → RSS 6, MSE 1.5 가 나와야 합니다
hx = np.array([1, 2, 3, 4])
hy = np.array([3, 5, 4, 8])
print('ŷ = 2x 일 때 RSS =', rss(hy, predict(hx, 2, 0)))
print('ŷ = 2x 일 때 MSE =', mse(hy, predict(hx, 2, 0)))

### a를 바꿔 가며 MSE 보기

절편 `b`를 0으로 고정하고 기울기 `a`만 바꾸면 MSE가 어떻게 변하는지 그려 봅니다.
**아래로 볼록한 포물선**이 나오고, 가장 낮은 점의 `a`가 가장 좋은 기울기입니다.

In [ ]:
a_list = np.linspace(0, 4, 81)
mse_list = [mse(hy, predict(hx, a, 0)) for a in a_list]

best = a_list[np.argmin(mse_list)]
print(f'MSE가 가장 작은 a ≈ {best:.2f},  그때 MSE ≈ {min(mse_list):.3f}')

plt.plot(a_list, mse_list)
plt.scatter([best], [min(mse_list)], color='red', zorder=3)
plt.xlabel('a (기울기)'); plt.ylabel('MSE')
plt.title('데이터 4개, ŷ = a·x 일 때 MSE(a)'); plt.show()

## 4. STEP 3-① · 공식으로 한 번에

RSS를 가장 작게 만드는 `a`, `b`는 공식으로 바로 구할 수 있습니다.

$$a = \frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}, \qquad b = \bar y - a\,\bar x$$

`x̄`, `ȳ`는 각각 x의 평균, y의 평균입니다. 공식을 외울 필요는 없습니다. 오른쪽에 **데이터(x, y)만** 있다는 점을 보세요.

In [ ]:
def fit_formula(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    # ✏️ 실습 ③ : 기울기 a 공식   힌트: np.sum((x - x_bar) * (y - y_bar)) / np.sum(…)
    a = 여기를_채우세요
    b = y_bar - a * x_bar
    return a, b


a_f, b_f = fit_formula(x_train, y_train)
print(f'공식으로 구한 직선:  ŷ = {a_f:.3f}·x + {b_f:.3f}')
print('실제로 정해 둔 값 :  y = 4.800·x + 7.000  (잡음 때문에 조금 다릅니다)')
print(f'훈련 MSE = {mse(y_train, predict(x_train, a_f, b_f)):.3f}')

# 넘파이의 함수와 같은 답인지 확인
print('np.polyfit 결과   :', np.round(np.polyfit(x_train, y_train, 1), 3))

xs = np.linspace(0, 3, 50)
plt.scatter(x_train, y_train, s=18, label='훈련 데이터')
plt.plot(xs, predict(xs, a_f, b_f), color='red', label='공식으로 찾은 직선')
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)')
plt.legend(); plt.show()

### 같은 공식을 행렬로: 정규방정식

입력이 여러 개(TV, 라디오, 신문…)가 되면 위 공식을 하나씩 늘려 쓰기 어렵습니다.
그래서 데이터를 **행렬**로 묶어 한 줄로 씁니다.

$$\mathbf{X} = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{bmatrix},\quad
\boldsymbol\beta = \begin{bmatrix} b \\ a \end{bmatrix},\quad
\hat{\mathbf{Y}} = \mathbf{X}\boldsymbol\beta$$

- 첫 열의 **1**은 절편 `b`를 곱하기 위한 칸입니다.
- RSS = (Y − Xβ)ᵀ(Y − Xβ) 를 β로 미분해서 0으로 놓고 풀면 **정규방정식**(Normal Equation)이 나옵니다.

$$\boldsymbol\beta = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{Y}$$

입력이 하나일 때는 위의 `a`, `b` 공식과 **똑같은 답**이 나와야 합니다.

In [ ]:
def fit_normal_eq(x, y):
    X = np.column_stack([np.ones(len(x)), x])     # [1, x] 를 열로 붙인 행렬 (x가 한 열이면 n × 2)
    # ✏️ 실습 ⑥ : β = (XᵀX)⁻¹ XᵀY   힌트: np.linalg.inv(…),  행렬 곱은 @,  전치는 X.T
    beta = 여기를_채우세요
    return beta                                   # [b, a] 순서


beta = fit_normal_eq(x_train, y_train)
print(f'정규방정식:  b = {beta[0]:.3f},  a = {beta[1]:.3f}')
print(f'a, b 공식  :  b = {b_f:.3f},  a = {a_f:.3f}   ← 같아야 합니다')
print()
print('X의 모양:', np.column_stack([np.ones(len(x_train)), x_train]).shape, ' → 데이터 80개 × (1, x)')
print('참고: 실무에서는 역행렬 대신 np.linalg.solve(X.T @ X, X.T @ y) 나 np.linalg.lstsq 를 씁니다. 더 정확하고 빠릅니다.')

### 실제 데이터로 풀어 보기: Advertising

지금까지는 정답을 정해 두고 만든 데이터였습니다. 이번에는 **실제로 조사한 데이터**에 같은 `fit_normal_eq`를 씁니다.

ISLR 교재(*An Introduction to Statistical Learning*)의 **Advertising 데이터**입니다. 시장 200곳의 매체별 광고비와 판매량이 들어 있습니다.

| 열 | 뜻 | 단위 |
|---|---|---|
| `TV`, `radio`, `newspaper` | TV · 라디오 · 신문 광고비 | 천 달러 |
| `sales` | 판매량 | 천 개 |

In [ ]:
import pandas as pd

# 수업 저장소에 올려 둔 사본을 먼저 읽고, 안 되면 교재 사이트에서 읽습니다
URLS = ['https://raw.githubusercontent.com/sejong-truelightn/ai-general-2026/main/data/Advertising.csv',
        'https://www.statlearning.com/s/Advertising.csv']
for url in URLS:
    try:
        ad = pd.read_csv(url, index_col=0)
        break
    except Exception:
        print('읽기 실패:', url)

print('데이터 모양:', ad.shape, ' → 시장 200곳 × (TV, 라디오, 신문, 판매량)')
ad.head()

In [ ]:
sales = ad['sales'].values

# 입력이 TV 하나: X = [1, TV]  (200 × 2)
tv = ad['TV'].values
beta_tv = fit_normal_eq(tv, sales)
a_ad, b_ad = fit_formula(tv, sales)
print(f'정규방정식:  b = {beta_tv[0]:.3f},  a = {beta_tv[1]:.4f}')
print(f'a, b 공식  :  b = {b_ad:.3f},  a = {a_ad:.4f}   ← 같아야 합니다')
print()

# 입력이 세 개: 같은 함수에 열 세 개를 그대로 넣습니다. X = [1, TV, 라디오, 신문]  (200 × 4)
X3 = ad[['TV', 'radio', 'newspaper']].values
beta3 = fit_normal_eq(X3, sales)
print('X의 모양:', np.column_stack([np.ones(len(X3)), X3]).shape)
print(f'정규방정식:  절편 = {beta3[0]:.3f},  TV = {beta3[1]:.4f},  라디오 = {beta3[2]:.4f},  신문 = {beta3[3]:.4f}')
print('np.linalg.lstsq 결과:', np.round(np.linalg.lstsq(np.column_stack([np.ones(len(X3)), X3]), sales, rcond=None)[0], 4))

`np.column_stack`은 `x`가 열 여러 개짜리 표여도 앞에 1 열을 붙여 줍니다. 그래서 **함수를 고치지 않고** 입력 세 개를 그대로 풀 수 있습니다.

계수 읽는 법 (다른 두 매체의 광고비는 그대로 둘 때):

- **TV ≈ 0.046**: TV 광고비를 1천 달러 늘리면 판매량이 약 0.046천 개, 즉 46개쯤 늘어납니다.
- **라디오 ≈ 0.19**: 같은 1천 달러로 약 190개가 늘어 TV보다 효과가 큽니다.
- **신문 ≈ −0.001**: 거의 0입니다. TV와 라디오를 고려하고 나면 신문 광고비는 판매량을 설명하는 데 거의 도움이 되지 않습니다.

## 5. STEP 3-② · 경사하강법으로 조금씩

공식이 없는 복잡한 모델은 **아무 값에서 시작해 MSE가 줄어드는 쪽으로 조금씩** 움직여 답을 찾습니다.

한 걸음의 규칙:

$$a \leftarrow a - \eta \times (\text{a 방향 기울기}), \qquad b \leftarrow b - \eta \times (\text{b 방향 기울기})$$

- `η` (에타, 학습률): 한 걸음의 크기. **사람이 정하는 값**(하이퍼파라미터)입니다.
- 기울기는 아래 코드에서 계산해 줍니다. MSE를 a, b로 미분한 값입니다.

In [ ]:
def fit_gd(x, y, lr=0.05, steps=2000):
    a, b = 0.0, 0.0                       # 아무 값에서 시작
    history = []                          # 걸음마다 MSE 기록
    for _ in range(steps):
        error = y - predict(x, a, b)      # 오차 = 정답 − 예측
        grad_a = -2 * np.mean(x * error)  # MSE를 a로 미분한 값
        grad_b = -2 * np.mean(error)      # MSE를 b로 미분한 값
        # ✏️ 실습 ④ : 기울기의 반대쪽으로 한 걸음  (a 와 b 둘 다)
        a = 여기를_채우세요
        b = 여기를_채우세요
        history.append(mse(y, predict(x, a, b)))
    return a, b, history


a_g, b_g, hist = fit_gd(x_train, y_train)
print(f'경사하강법:  ŷ = {a_g:.3f}·x + {b_g:.3f}')
print(f'공식      :  ŷ = {a_f:.3f}·x + {b_f:.3f}   ← 거의 같아야 합니다')

plt.plot(hist)
plt.yscale('log')
plt.xlabel('걸음 수'); plt.ylabel('훈련 MSE (로그 눈금)')
plt.title('경사하강법: 걸음마다 MSE가 줄어든다'); plt.show()

### 학습률 η 바꿔 보기

같은 데이터, 같은 출발점에서 `η`만 바꿉니다.

- 너무 작으면: 방향은 맞지만 **느립니다**.
- 적당하면: 금방 바닥에 도착합니다.
- 너무 크면: 바닥을 건너뛰어 **발산**합니다 (MSE가 폭발).

In [ ]:
for lr in [0.005, 0.05, 0.3]:
    with np.errstate(over='ignore', invalid='ignore'):
        _, _, h = fit_gd(x_train, y_train, lr=lr, steps=200)
    plt.plot(h, label=f'η = {lr}')
    print(f'η = {lr:<6} → 200걸음 후 MSE = {h[-1]:.4g}')

plt.yscale('log'); plt.ylim(1, 1e4)
plt.xlabel('걸음 수'); plt.ylabel('훈련 MSE (로그 눈금)')
plt.legend(); plt.title('학습률에 따른 차이'); plt.show()

## 6. STEP 4 · 예측과 평가

학습이 끝났으니 **처음 보는 테스트 데이터**로 채점합니다.

- **RMSE** = √MSE. 원래 단위(억 원)로 "평균 이만큼 빗나간다"고 읽을 수 있습니다.
- **R²** = 1 − RSS / TSS. 직선이 매출의 변화를 몇 %나 설명하는지입니다.
  - RSS = Σ(y − ŷ)²  (우리 모델의 오차 제곱합)
  - TSS = Σ(y − ȳ)²  (평균 하나로만 찍었을 때의 오차 제곱합)
  - 1에 가까울수록 좋고, 0이면 평균으로 찍는 것과 같습니다.

In [ ]:
def r2(y, y_hat):
    rss = np.sum((y - y_hat) ** 2)
    tss = np.sum((y - y.mean()) ** 2)
    # ✏️ 실습 ⑤ : R² = 1 − RSS / TSS
    return 여기를_채우세요


y_hat_test = predict(x_test, a_f, b_f)
print(f'테스트 RMSE = {np.sqrt(mse(y_test, y_hat_test)):.3f} 억 원')
print(f'테스트 R²   = {r2(y_test, y_hat_test):.3f}')
print()
print(f'광고비 2.5백만 원을 쓰면 예상 매출 = {predict(2.5, a_f, b_f):.2f} 억 원')

### 훈련 점수 vs 테스트 점수

훈련 데이터로 잰 점수와 테스트 데이터로 잰 점수를 나란히 봅니다.
직선처럼 단순한 모델은 두 값이 비슷하게 나옵니다.

In [ ]:
print(f'훈련 MSE  = {mse(y_train, predict(x_train, a_f, b_f)):.3f}')
print(f'테스트 MSE = {mse(y_test,  predict(x_test,  a_f, b_f)):.3f}')
print()
print('잡음 ε의 분산은 1.2² = 1.44 로 정해 두었습니다.')
print('어떤 모델도 평균적으로 이보다 크게 낮출 수는 없습니다 (줄일 수 없는 오차).')

---

## 도전 과제

아래 코드는 위에서 만든 `x_train`, `y_train` 등을 **바꾸지 않습니다**. 조건을 바꾼 데이터를 새로 만들어 비교합니다.
먼저 데이터 만들기 과정을 함수로 묶어 둡니다. 1번 셀과 똑같은 순서로 만들기 때문에 기본값이면 같은 데이터가 나옵니다.

도전 과제 코드에도 `여기를_채우세요` 가 있습니다. 위의 빈칸 ①~⑥을 먼저 끝낸 뒤 풀어 보세요.

In [ ]:
def make_data(N=100, noise_sd=1.2, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 3, N)
    y = 7 + 4.8 * x + rng.normal(0, noise_sd, N)
    idx = rng.permutation(N)
    n_train = int(N * 0.8)
    train, test = idx[:n_train], idx[n_train:]
    return x[train], y[train], x[test], y[test]


# 확인: 기본값이면 1번 셀의 데이터와 같아야 합니다
xt, yt, _, _ = make_data()
print('1번 셀과 같은 데이터인가?', np.allclose(xt, x_train) and np.allclose(yt, y_train))

### 1. 잡음 키우기

잡음의 표준편차를 `1.2`에서 `3`으로 키웁니다. 기울기는 여전히 4.8 근처로 찾아지나요? R²는 어떻게 변하나요?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, sd in zip(axes, [1.2, 3]):
    xtr, ytr, xte, yte = make_data(noise_sd=sd)
    # ✏️ 도전 1 : 위에서 만든 함수로 xtr, ytr 에 맞는 a, b 를 구한다
    a, b = 여기를_채우세요
    print(f'잡음 표준편차 {sd:<3} →  ŷ = {a:.3f}·x + {b:.3f},  테스트 R² = {r2(yte, predict(xte, a, b)):.3f}')

    ax.scatter(xtr, ytr, s=14)
    ax.plot(xs, predict(xs, a, b), color='red')
    ax.set_title(f'잡음 표준편차 = {sd}')
    ax.set_xlabel('TV 광고비 (백만 원)')

axes[0].set_ylabel('매출 (억 원)')
plt.show()

### 2. 이상치 하나

훈련 데이터에 광고비 2.9, 매출 0인 점을 하나 넣고 다시 학습합니다. 직선이 얼마나 끌려가나요?

In [ ]:
# ✏️ 도전 2 : 훈련 데이터 끝에 (광고비 2.9, 매출 0) 인 점을 붙인다   힌트: np.append(배열, 값)
x_out = 여기를_채우세요
y_out = 여기를_채우세요

a_o, b_o = fit_formula(x_out, y_out)
print(f'이상치 없음 :  ŷ = {a_f:.3f}·x + {b_f:.3f}')
print(f'이상치 1개  :  ŷ = {a_o:.3f}·x + {b_o:.3f}   ← 점 하나로 기울기가 {a_f - a_o:.2f} 줄었습니다')

# 이상치 한 점이 훈련 MSE에서 차지하는 몫
err2 = (y_out - predict(x_out, a_o, b_o)) ** 2
print(f'이상치 한 점이 오차 제곱합에서 차지하는 비율 = {err2[-1] / err2.sum():.0%}  (점 81개 중 1개)')

plt.scatter(x_train, y_train, s=14, label='훈련 데이터')
plt.scatter([2.9], [0], color='black', marker='x', s=80, label='이상치')
plt.plot(xs, predict(xs, a_f, b_f), color='red', label='이상치 없음')
plt.plot(xs, predict(xs, a_o, b_o), color='orange', linestyle='--', label='이상치 포함')
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)')
plt.legend(); plt.show()

### 3. 데이터 개수

데이터를 10개만 모았다고 하고, seed를 바꿔 가며 20번 학습합니다. 찾은 기울기가 얼마나 흔들리나요? 100개일 때와 비교합니다.

In [ ]:
for N_small in [10, 100]:
    slopes = []
    for seed in range(20):
        # ✏️ 도전 3 : 데이터 N_small 개를 seed 로 만든다   힌트: make_data(N=…, seed=…)
        xtr, ytr, _, _ = 여기를_채우세요
        a, _ = fit_formula(xtr, ytr)
        slopes.append(a)
    slopes = np.array(slopes)
    print(f'N = {N_small:<3} →  기울기 범위 {slopes.min():.2f} ~ {slopes.max():.2f},  표준편차 {slopes.std():.3f}')
    plt.scatter(np.full(20, N_small), slopes, s=18, alpha=0.7)

plt.axhline(4.8, color='gray', linestyle='--', label='실제 기울기 4.8')
plt.xscale('log'); plt.xticks([10, 100], ['N = 10', 'N = 100'])
plt.ylabel('찾은 기울기 a'); plt.title('seed 20개로 학습한 기울기')
plt.legend(); plt.show()

### 4. 곡선 모델

`np.polyfit`으로 차수 1, 3, 9 다항식을 학습하고 훈련 MSE와 테스트 MSE를 비교합니다.
훈련 데이터가 80개일 때와 10개일 때를 함께 봅니다.

In [ ]:
xtr10, ytr10, _, _ = make_data(N=13, seed=1)   # 훈련 10개, 테스트 3개
xtr10, ytr10 = xtr10[:10], ytr10[:10]

for name, xtr, ytr in [('훈련 80개', x_train, y_train), ('훈련 10개', xtr10, ytr10)]:
    print(f'[{name}]  테스트는 위의 x_test 20개로 같게 잽니다')
    for deg in [1, 3, 9]:
        # ✏️ 도전 4 : deg 차 다항식을 학습하고 테스트 MSE 를 잰다   힌트: np.polyfit(x, y, 차수), np.polyval(계수, x)
        coef = 여기를_채우세요
        tr = mse(ytr, np.polyval(coef, xtr))
        te = 여기를_채우세요
        print(f'  차수 {deg}:  훈련 MSE = {tr:7.3f}   테스트 MSE = {te:.4g}')
    print()

# 훈련 10개로 학습한 곡선 그리기
for deg in [1, 3, 9]:
    coef = np.polyfit(xtr10, ytr10, deg)
    plt.plot(xs, np.polyval(coef, xs), label=f'차수 {deg}')
plt.scatter(xtr10, ytr10, color='black', s=24, zorder=3, label='훈련 10개')
plt.scatter(x_test, y_test, color='gray', marker='s', s=14, label='테스트')
plt.ylim(0, 30)
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)')
plt.legend(); plt.title('훈련 10개로 학습한 다항식'); plt.show()

### 5. 입력이 여러 개 (Advertising 데이터)

4절에서 불러온 Advertising 데이터를 훈련 160곳, 테스트 40곳으로 나누고, 입력을 하나씩 늘려 가며 테스트 R²를 비교합니다.
라디오를 넣으면 R²가 얼마나 오르나요? 신문까지 넣으면 어떻게 되나요?

In [ ]:
rng3 = np.random.default_rng(0)
idx3 = rng3.permutation(len(ad))
tr3, te3 = idx3[:160], idx3[160:]          # 훈련 160곳 / 테스트 40곳
sales = ad['sales'].values

for cols in [['TV'], ['TV', 'radio'], ['TV', 'radio', 'newspaper']]:
    # ✏️ 도전 5 : [1, 고른 열들] 을 붙인 행렬   힌트: np.column_stack([np.ones(len(ad)), …]),  고른 열은 ad[cols].values
    X = 여기를_채우세요
    # ✏️ 도전 5 : 정규방정식 β = (XᵀX)⁻¹ XᵀY  — 훈련 행만 쓴다: X[tr3], sales[tr3]
    beta_c = 여기를_채우세요
    y_hat = X[te3] @ beta_c
    print(f'{" + ".join(cols)}')
    print(f'   계수 = {np.round(beta_c, 4)}')
    print(f'   테스트 RMSE = {np.sqrt(mse(sales[te3], y_hat)):.3f} 천 개,  테스트 R² = {r2(sales[te3], y_hat):.3f}')